# 🍷 Principal Component Analysis (PCA) — Wine Dataset

---

## What is PCA?
**Principal Component Analysis (PCA)** is an **unsupervised** dimensionality reduction technique.
It projects high-dimensional data into a lower-dimensional space while retaining
as much **variance (information)** as possible.

Think of it as finding the "best angle" to view your data so that the most
meaningful differences between data points are preserved.

---

## About the Dataset
| Property | Detail |
|---|---|
| **Source** | `wine.csv` |
| **Samples** | 178 wine samples |
| **Features** | 13 chemical measurements (Alcohol, Ash, Magnesium, etc.) |
| **Label** | `Wine_type` — 3 cultivars (grape varieties): **1, 2, 3** |

---

## Workflow
1. 📥 Load the data
2. 🔍 Explore features and labels
3. ⚖️ Normalize (Standardize) the features
4. 📉 Apply PCA (reduce to 2 components)
5. 📊 Visualize the results

---
## Step 0 — Install & Import Libraries

We use four core libraries:
- **NumPy** — numerical operations and array handling
- **Pandas** — loading and organizing data in table (DataFrame) form
- **Matplotlib** — plotting and visualization
- **Scikit-learn** — StandardScaler (normalization) and PCA (dimensionality reduction)

In [ ]:
import numpy as np                          # For numerical operations
import pandas as pd                         # For loading and managing the dataset
import matplotlib.pyplot as plt             # For plotting the PCA scatter chart

from sklearn.preprocessing import StandardScaler   # For normalizing feature values
from sklearn.decomposition import PCA              # For reducing dimensions via PCA

print('All libraries imported successfully!')

---
## Step 1 — Load the Dataset

The dataset is stored in a CSV file (`wine.csv`).

**If running in Google Colab**, you need to upload the file first:
- Click the 📁 folder icon in the left sidebar
- Click the upload button and select `wine.csv`
- Then run the cell below

The dataset contains **178 rows** (wine samples) and **14 columns**
(13 chemical features + 1 label column `Wine_type`).

In [ ]:
# ----- Optional: Upload file if running in Google Colab -----
# Uncomment the 3 lines below if you need to upload wine.csv from your computer
# from google.colab import files
# uploaded = files.upload()    # A dialog will pop up to choose your file
# print('File uploaded!')

# Load the CSV file into a Pandas DataFrame
# A DataFrame is like a spreadsheet — rows are samples, columns are features
wine_dataset = pd.read_csv('wine.csv')

# Confirm the data loaded correctly by checking its shape (rows, columns)
print('Dataset shape:', wine_dataset.shape)   # Expected: (178, 14)

# Preview the first 5 rows to understand the data structure
wine_dataset.head()

---
## Step 2 — Explore Features and Labels

Before any analysis, it's important to understand what we're working with:
- **Features** = the 13 chemical measurements used as input to PCA
- **Label** = `Wine_type` (1, 2, or 3) — the grape cultivar, used only for coloring the final plot

> 📝 Note: PCA is **unsupervised** — it does NOT use the labels during computation.
> Labels are only used afterward to color-code the scatter plot.

In [ ]:
# Automatically identify all feature columns (everything except the label column)
# This approach is flexible — it works even if columns are reordered
features = [col for col in wine_dataset.columns if col != 'Wine_type']

print('Number of features:', len(features))   # Expected: 13
print('Feature names:', features)

print()

# Check how many samples belong to each cultivar (class balance)
# Imbalanced classes can sometimes affect how well PCA separates groups visually
print('Label distribution (Wine_type):')
print(wine_dataset['Wine_type'].value_counts().sort_index())
# Cultivar 1: 59 samples | Cultivar 2: 71 samples | Cultivar 3: 48 samples

---
## Step 3 — Normalize (Standardize) the Features

### Why is normalization necessary?
PCA works by finding directions of **maximum variance** in the data.
If features have very different scales, PCA will be dominated by the
features with the largest numerical range — even if they aren't the most informative.

**Example from this dataset:**
- `Proline` ranges from ~278 to ~1680
- `Nonflavanoid.phenols` ranges from ~0.13 to ~0.66

Without scaling, `Proline` would overwhelm everything else.

### What does StandardScaler do?
`StandardScaler` transforms each feature so that:
- **Mean = 0** (centers the data)
- **Std = 1** (puts all features on equal footing)

Formula: `z = (x - mean) / std`

In [ ]:
# Extract only the feature columns (exclude the label 'Wine_type')
# wine_dataset[features] gives us a 178 x 13 table of raw measurements
raw_features = wine_dataset[features]

# Apply StandardScaler:
#   fit_transform() = learn the mean & std from the data, then apply the transformation
# Result: x is a 178 x 13 NumPy array with each feature having mean=0 and std=1
x = StandardScaler().fit_transform(raw_features)

# Verify normalization worked correctly
# Mean should be extremely close to 0, Std should be exactly 1.0
print('Normalized data stats:')
print('  Mean : {:.6f}  (should be ~0)'.format(np.mean(x)))
print('  Std  : {:.6f}  (should be ~1)'.format(np.std(x)))
print('  Shape:', x.shape)   # Should be (178, 13)

---
## Step 4 — Apply PCA

### How PCA works (simplified):
1. PCA finds the directions (called **principal components**) in the data along which the data varies the most
2. **PC1** = the direction of greatest variance
3. **PC2** = the direction of second greatest variance, perpendicular to PC1
4. All 178 wine samples are then re-expressed using just these 2 coordinates

### Why only 2 components?
- 2 components allow us to plot the data on a standard 2D scatter plot
- The **explained variance ratio** tells us how much information is retained
- More components = more information retained, but harder to visualize

> 💡 **Choosing the number of components** is part art, part science.
> Common approaches: scree plot, eigenvalues > 1, or explained variance threshold (e.g., 80%).

In [ ]:
# Initialize PCA and reduce from 13 dimensions → 2 principal components
# n_components=2 means we keep only the 2 most informative directions
pca = PCA(n_components=2)

# fit_transform() does two things:
#   fit()      = learns the principal component directions from the data
#   transform()= projects all 178 samples onto those 2 new axes
# Result: a 178 x 2 array — each sample now has just 2 coordinates (PC1, PC2)
principal_components = pca.fit_transform(x)

print('PCA output shape:', principal_components.shape)  # Expected: (178, 2)

# Wrap the PCA output in a DataFrame for easy indexing and plotting
# Each row is one wine sample; columns are its position on PC1 and PC2
pca_df = pd.DataFrame(principal_components, columns=['PC1', 'PC2'])

print()
print('First 5 rows of PCA result:')
print(pca_df.head())

---
## Step 4b — Explained Variance Ratio

The **explained variance ratio** tells us what fraction of the total information
in the original 13-feature dataset is captured by each principal component.

- A higher ratio = that component carries more of the original signal
- Together, PC1 + PC2 give us the **combined retention rate**

> If the combined ratio is above ~50–60%, the 2D PCA plot is a reasonably
> faithful representation of the original high-dimensional data.

In [ ]:
# explained_variance_ratio_ gives the proportion of variance captured by each component
# Index [0] = PC1, Index [1] = PC2
pc1_var, pc2_var = pca.explained_variance_ratio_

print('Explained variance per principal component:')
print('  PC1 : {:.1%}  — direction of greatest spread in the data'.format(pc1_var))
print('  PC2 : {:.1%}  — direction of second greatest spread'.format(pc2_var))
print('  ──────────────────────────────────────────────')
print('  Total: {:.1%}  of original information retained in 2D'.format(pc1_var + pc2_var))

---
## Step 5 — Visualize the PCA Result

We now plot each wine sample as a point in 2D space using its PC1 and PC2 values.

**Color coding by cultivar:**
- 🔴 Red = Cultivar 1 (59 samples)
- 🟢 Green = Cultivar 2 (71 samples)
- 🔵 Blue = Cultivar 3 (48 samples)

**What to look for:**
- **Well-separated clusters** → PCA successfully captured the differences between cultivars
- **Overlapping clusters** → the cultivars are chemically similar along these two axes

> Remember: PCA did NOT use the labels — any separation is purely driven by
> the chemical composition of the wines.

In [ ]:
# Set up the figure canvas (10x10 inches for a clear, large plot)
plt.figure(figsize=(10, 10))

# Axis labels and title
plt.xlabel('Principal Component 1  ({:.1%} variance)'.format(pc1_var), fontsize=20)
plt.ylabel('Principal Component 2  ({:.1%} variance)'.format(pc2_var), fontsize=20)
plt.title('PCA of Wine Dataset — 3 Cultivars', fontsize=22)
plt.xticks(fontsize=12)
plt.yticks(fontsize=14)

# Define one color and display name per cultivar
# Dictionary format: {label: (color_code, legend_name)}
colors = {
    1: ('r', 'Cultivar 1'),   # Red   — 59 samples
    2: ('g', 'Cultivar 2'),   # Green — 71 samples
    3: ('b', 'Cultivar 3')    # Blue  — 48 samples
}

# Loop through each cultivar and plot its points separately
# This allows each group to have its own color in the legend
for label, (color, name) in colors.items():

    # Create a boolean mask: True for rows where Wine_type matches the current label
    mask = wine_dataset['Wine_type'] == label

    # Plot only the PC1/PC2 values for this cultivar's samples
    plt.scatter(
        pca_df.loc[mask, 'PC1'],   # x-axis: position along PC1
        pca_df.loc[mask, 'PC2'],   # y-axis: position along PC2
        c=color,                   # dot color
        s=50,                      # dot size
        label=name                 # legend entry
    )

# Add legend, tighten layout, and display
plt.legend(prop={'size': 15})
plt.tight_layout()
plt.savefig('pca_wine.png', dpi=150)   # Save a high-resolution copy
plt.show()

print('Plot saved as pca_wine.png')

---
## Summary & Interpretation

| Component | Variance Explained | Role |
|---|---|---|
| **PC1** | ~36% | Direction of greatest chemical variation |
| **PC2** | ~19% | Direction of second greatest chemical variation |
| **Combined** | ~55% | Total information retained in the 2D plot |

### What the plot tells us:
- The **three cultivars form distinct clusters** — meaning the 13 chemical features
  genuinely differ between grape varieties, and PCA was able to surface that structure
- **Cultivar 3 (blue)** tends to sit apart from the others along PC1
- **Cultivars 1 and 2** have more overlap but are still largely separable

### Key Takeaways:
> ✅ PCA successfully reduced 13 chemical dimensions to 2, while preserving ~55% of the variance
>
> ✅ The clear cluster separation confirms that wine cultivar identity is encoded in the chemistry
>
> ✅ This 2D representation could serve as input to a downstream classifier (e.g., KNN, SVM)

---
*Note: 0 = Benign, 1 = Malignant applies to the breast cancer dataset.
For wine: 1, 2, 3 represent three different grape cultivar origins.*